# Agentic Mobility - Enhanced Visualization Demo

This notebook demonstrates the **new enhanced visualization functions** with better control over legends, filtering, and customization.

## Key Features

1. **Legend Control**: Hide/show legends with `legend` parameter
2. **UE Filtering**: Plot specific UEs with `ue_ids` parameter
3. **Better Performance**: Optimized for large datasets
4. **More Customization**: Control figure size, titles, markers, etc.

## Setup

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Add the maveric root directory to the path
maveric_root = Path.cwd().parent.parent.parent.parent
if str(maveric_root) not in sys.path:
    sys.path.insert(0, str(maveric_root))

print(f"Maveric root: {maveric_root}")
print(f"Current working directory: {Path.cwd()}")

In [ ]:
# Import the NEW enhanced visualization functions
from radp.digital_twin.agentic_mobility.visualization.tracks import (
    plot_ue_tracks,
    plot_ue_tracks_comparison
)

from radp.digital_twin.agentic_mobility.visualization import utils

print("Successfully imported NEW enhanced visualization functions!")

## Load Data

In [ ]:
# Define the data directory
data_dir = Path.cwd() / "generated_ues"

# List all available CSV files
csv_files = list(data_dir.glob("*.csv"))

print(f"Data directory: {data_dir}")
print(f"\nAvailable CSV files ({len(csv_files)}):")
for i, csv_file in enumerate(csv_files, 1):
    print(f"  {i}. {csv_file.name}")

if not csv_files:
    print("\n⚠️  No CSV files found!")
    print("Please run 'python radp/digital_twin/agentic_mobility/examples/end_to_end_example.py' first.")

In [ ]:
# Load the first CSV file
if csv_files:
    selected_csv = csv_files[0]
    print(f"Loading: {selected_csv.name}")
    
    df = pd.read_csv(selected_csv)
    
    # Load metadata if available
    metadata_file = selected_csv.with_name(selected_csv.stem + "_metadata.json")
    if metadata_file.exists():
        with open(metadata_file, 'r') as f:
            metadata = json.load(f)
        print(f"Loaded metadata: {metadata_file.name}")
    else:
        metadata = None
        print("No metadata file found")
    
    print(f"\nDataset info:")
    print(f"  - Shape: {df.shape}")
    print(f"  - UEs: {df['mock_ue_id'].nunique()}")
    print(f"  - Ticks: {df['tick'].max() + 1 if len(df) > 0 else 0}")
    print(f"  - Columns: {list(df.columns)}")

## Example 1: Plot ALL UEs WITHOUT Legend

**Use Case**: When you have many UEs and don't need individual labels.

**Solution**: Use `legend=False` (default)

In [ ]:
if csv_files:
    %matplotlib inline
    
    print("Plotting all UEs without legend...")
    plot_ue_tracks(df, legend=False)

## Example 2: Plot Specific UEs WITH Legend

**Use Case**: Focus on a subset of UEs for detailed analysis.

**Solution**: Use `ue_ids` parameter to filter + `legend=True`

In [ ]:
if csv_files:
    # Plot only the first 5 UEs
    print("Plotting first 5 UEs with legend...")
    
    # Get the first 5 unique UE IDs
    first_5_ues = sorted(df['mock_ue_id'].unique())[:5]
    print(f"Selected UE IDs: {first_5_ues}")
    
    plot_ue_tracks(
        df,
        legend=True,
        ue_ids=first_5_ues,
        title="First 5 UEs - Detailed View",
        show_end_points=False,  # Also show where they end
    )

## Example 3: Plot a Single UE Track

**Use Case**: Analyze individual UE movement in detail.

**Solution**: Filter to a single UE ID

In [ ]:
if csv_files:
    # Plot just one UE
    single_ue_id = df['mock_ue_id'].unique()[0]
    print(f"Plotting single UE: {single_ue_id}")
    
    plot_ue_tracks(
        df,
        legend=True,
        ue_ids=[single_ue_id],
        title=f"Detailed Track for UE {single_ue_id}",
        figsize=(10, 8),
        show_end_points=True,
    )

## Example 4: Comparison of Two Datasets

**Use Case**: Compare mobility patterns from different scenarios.

**Solution**: Use `plot_ue_tracks_comparison()`

In [ ]:
if len(csv_files) >= 2:
    # Load second dataset
    df2 = pd.read_csv(csv_files[1])
    
    print(f"Comparing:")
    print(f"  Dataset 1: {csv_files[0].name}")
    print(f"  Dataset 2: {csv_files[1].name}")
    
    # Compare first 5 UEs from both datasets
    common_ue_ids = sorted(set(df['mock_ue_id'].unique()) & set(df2['mock_ue_id'].unique()))[:5]
    
    if common_ue_ids:
        print(f"  Common UE IDs to compare: {common_ue_ids}")
        plot_ue_tracks_comparison(
            df,
            df2,
            legend=True,
            ue_ids=None,
            titles=("Scenario 1", "Scenario 2"),
        )
    else:
        print("  No common UE IDs found, comparing all UEs without filtering")
        plot_ue_tracks_comparison(
            df,
            df2,
            legend=False,
            titles=(csv_files[0].stem, csv_files[1].stem),
        )
else:
    print("Need at least 2 datasets for comparison.")
    print("Run end_to_end_example.py multiple times to generate more datasets.")

## Example 6: Customization Options

Demonstrate various customization options available.

In [ ]:
if csv_files:
    # Custom figure size, thicker arrows, different title
    subset_ues = sorted(df['mock_ue_id'].unique())[:8]
    
    plot_ue_tracks(
        df,
        legend=True,
        ue_ids=subset_ues,
        figsize=(16, 10),  # Larger figure
        title="Custom Visualization - 8 UEs",
        show_start_points=True,
        show_end_points=True,
        arrow_width=0.003,  # Thicker arrows for better visibility
    )

## Example 7: Data Analysis with Filtering

Use pandas filtering before visualization for advanced analysis.

In [ ]:
if csv_files:
    
    # Calculate distance for each UE
    ue_distances = df.groupby('mock_ue_id').apply(utils.calculate_total_distance).sort_values(ascending=False)
    
    print("Top 5 most mobile UEs:")
    print(ue_distances.head())
    
    # Plot the top 5 most mobile UEs
    top_5_mobile = ue_distances.head(5).index.tolist()
    plot_ue_tracks(
        df,
        ue_ids=top_5_mobile,
        title="Top 5 Most Mobile UEs",
    )